# Lichess extract (free, on Google Colab)

Streams Lichess months (open database, CC0) on Google's computer, keeps players who play both bullet and slow games, computes per-move measures with anonymous IDs, checks a sample of positions with the Stockfish engine, and saves **small anonymised files** to your Google Drive (`compression-learner-data/lichess/`).

- Nothing is downloaded to your own computer. Raw data is streamed and never stored; temporary files vanish when the session ends.
- Player names become anonymous IDs made with your secret key; a safety check stops before saving if any name is left.
- **Secret key:** the same as before. Easiest: Colab's **key icon** (left sidebar) → secret `LICHESS_ID_SALT` with *Notebook access* on; otherwise you'll be asked to type it. The fingerprint check stops the run if it differs from last time.
- **Months:** players are picked from the first month; all months are processed in order, so each player's history carries over (June = exploratory, July = confirmation).
- **First:** run with `TEST_RUN = True` (about 15 min) to check everything works. Then set `TEST_RUN = False` for the full run (several hours; keep this tab open).
- **How to run:** Runtime → Run all, allow Drive access.
- **Output:** big files are split by player into parts of about 25 MB, so they can be uploaded to a chat.

Code: https://github.com/saurabh9729/compression-learner-data (`pipelines/lichess/`)

In [ ]:
# Settings
TEST_RUN = True                     # True: quick check on the first 1M games per month and 500 engine positions
MONTHS = ["2018-06", "2018-07"]     # players are picked from the first month
MIN_BULLET = 30                     # player needs at least this many bullet games ...
MIN_SLOW = 15                       # ... and this many rapid + classical games (in the first month)
N_PLAYERS = 300                     # random sample of eligible players (same seed -> same players as before)
ENGINE_N = 20_000                   # similar-position moves checked by Stockfish
ENGINE_DEPTH = 12
MAX_GAMES = 1_000_000 if TEST_RUN else 0
if TEST_RUN:
    ENGINE_N = 500

In [ ]:
# Install tools and fetch the pipeline code from the public repo
!apt-get -qq install -y zstd stockfish > /dev/null
!pip -q install chess pyarrow
import os, getpass
BASE = "https://raw.githubusercontent.com/saurabh9729/compression-learner-data/main/pipelines/lichess"
os.makedirs("/content/work", exist_ok=True); os.chdir("/content/work")
for f in ["filter_lichess.py", "select_players.py", "lichess_pipeline.py", "check_no_names.py", "engine_eval.py"]:
    !curl -sSfLO {BASE}/{f}
!ls -l *.py
!ls -l /usr/games/stockfish

In [ ]:
# Secret key for the anonymous IDs, plus a check that it is the same key as last time
import hashlib
from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/compression-learner-data/lichess"
os.makedirs(dest, exist_ok=True)

key = None
try:
    from google.colab import userdata
    key = userdata.get("LICHESS_ID_SALT")       # from the Secrets panel (key icon), if set
except Exception:
    pass
if not key:
    key = getpass.getpass("Secret key: ")
assert len(key) >= 12, "use a longer key (12+ characters)"

# Fingerprint: 6 characters from a slow hash. Catches typos; too short to recover the key.
fp = hashlib.pbkdf2_hmac("sha256", key.encode(), b"lichess-key-check", 200_000).hex()[:6]
fp_file = os.path.join(dest, "key_fingerprint.txt")
if os.path.exists(fp_file):
    saved = open(fp_file).read().strip()
    assert fp == saved, f"STOP: this is not the same key as before (fingerprint {fp}, saved {saved})"
    print("key check: same key as before ✓", fp)
else:
    print("first run: key fingerprint", fp, "(saved to Drive once this run succeeds)")
os.environ["LICHESS_ID_SALT"] = key
URLS = [f"https://database.lichess.org/standard/lichess_db_standard_rated_{m}.pgn.zst" for m in MONTHS]
print("sources:", *URLS, sep="\n  ")

In [ ]:
# Pass 1: count games per player in the FIRST month (streamed; nothing stored but the counts)
!curl -sL {URLS[0]} | zstdcat 2>/dev/null | python filter_lichess.py --count counts.csv --max-games {MAX_GAMES}
!wc -l < counts.csv

In [ ]:
# Pick players who play both fast and slow games
!python select_players.py counts.csv --min-bullet {MIN_BULLET} --min-slow {MIN_SLOW} --n {N_PLAYERS} --out players.txt

In [ ]:
# Pass 2: keep only their games (with clock times), month after month
!rm -f filtered.pgn
for u in URLS:
    print("streaming", u)
    !curl -sL {u} | zstdcat 2>/dev/null | python filter_lichess.py --extract players.txt --max-games {MAX_GAMES} >> filtered.pgn
!ls -lh filtered.pgn

In [ ]:
# Per-move measures with anonymous IDs, then the safety check
OUT = "moves_" + "_".join(MONTHS) + ("_test" if TEST_RUN else "")
!python lichess_pipeline.py filtered.pgn --players players.txt --out {OUT}.parquet
!python check_no_names.py players.txt {OUT}.parquet {OUT}_players.csv

In [ ]:
# Engine check: Stockfish evaluates the best move, the played move, and the player's usual move under
# each stored representation, in a sample of similar-but-not-identical positions
!python engine_eval.py {OUT}.parquet --out {OUT}_engine.parquet --n {ENGINE_N} --depth {ENGINE_DEPTH}
!python check_no_names.py players.txt {OUT}_engine.parquet

In [ ]:
# Save only the anonymised files to Google Drive (runs only if the safety check passed).
# Big files are split by player into parts of about 25 MB, so they can be uploaded to a chat.
import subprocess, shutil, math, zlib
import pyarrow.parquet as pq, pyarrow.compute as pc
ok = subprocess.run(["python", "check_no_names.py", "players.txt", f"{OUT}.parquet", f"{OUT}_players.csv", f"{OUT}_engine.parquet"]).returncode == 0
assert ok, "safety check failed: nothing saved"
size_mb = os.path.getsize(f"{OUT}.parquet") / 1e6
n_parts = max(1, math.ceil(size_mb / 25))
saved = [f"{OUT}_players.csv", f"{OUT}_engine.parquet"]
for f in saved:
    shutil.copy(f, dest)
if n_parts == 1:
    shutil.copy(f"{OUT}.parquet", dest); saved.append(f"{OUT}.parquet")
else:
    t = pq.read_table(f"{OUT}.parquet")
    part_of = [zlib.crc32(p.encode()) % n_parts for p in t.column("player").to_pylist()]
    import pyarrow as pa
    part_col = pa.array(part_of)
    for k in range(n_parts):
        name = f"{OUT}_part{k + 1}of{n_parts}.parquet"
        pq.write_table(t.filter(pc.equal(part_col, k)), name)
        shutil.copy(name, dest); saved.append(name)
if not os.path.exists(fp_file):
    open(fp_file, "w").write(fp + "\n")
print(f"moves file {size_mb:.0f} MB -> {n_parts} part(s)")
print("saved to Drive:", dest)
for f in saved: print("  ", f, f"{os.path.getsize(os.path.join(dest, f)) / 1e6:.1f} MB")